# jtagent Colab (TAN Drive)

Run on a **T4 GPU**. After a factory reset, run cells **in order**. Mount Drive before any download or `makedirs`.

Folder: `1ondyw5YrwXpE6jV48nYpRlg4Z1QkZWUB` (TAN)

Notebook (this file): https://colab.research.google.com/drive/17lXE2XDv9LlKfDOLrKUs1YNEQFP95cDd

HF sources are ~500GB. Colab disk is ~200GB free. **Never download the full set onto `/content`.** Stream HF → Drive, then delete the local copy.

**The new 6** (also on this same notebook):
1. `jtagent/Qwen3.8-27B-Uncensored-MLX-bucket` (model)
2. `jtagent/jt-agent-model` (model)
3. `jtagent/jt-agent-data` (model)
4. `jtagent/jt-agent-dataset` (dataset)
5. `jtagent/chrome-browser-data` (dataset — do not print DB contents)
6. `jtagent/jt-agent-data` (dataset)

**Remove duplicates:** skip any file already on Drive with the same name and size. Training JSONL drops repeat `content_hash` rows.

Order:
1. Load Colab secret `jayti`
2. **Mount Drive** first
3. Confirm go4garage 6 shards + the new 6 folders
4. Segment / train GPT-2 LoRA (does not load 8B/27B weights)
5. Write adapters back to Drive


In [ ]:
from google.colab import userdata
import os

# Colab Secrets panel: name must be exactly `jayti`
os.environ["CURSOR_API_KEY"] = userdata.get("jayti")
print("cursor_key_loaded", bool(os.environ.get("CURSOR_API_KEY")))
print("cursor_key_name", "jayti")


In [ ]:
import shutil
from pathlib import Path
from google.colab import drive

# Mount FIRST. Do not mkdir /content/drive or download into it before this cell.
mp = Path("/content/drive")
mydrive = mp / "MyDrive"
if mp.exists() and not mydrive.exists():
    stale = Path("/tmp/drive_stale_local")
    if stale.exists():
        shutil.rmtree(stale, ignore_errors=True)
    shutil.move(str(mp), str(stale))
    shutil.rmtree(stale, ignore_errors=True)
    print("removed_stale_local_drive", True)
drive.mount("/content/drive")
print("MyDrive", mydrive.exists())
print("root", [p.name for p in mp.iterdir()][:20] if mp.exists() else [])

In [ ]:
from pathlib import Path

FOLDER_ID = "1ondyw5YrwXpE6jV48nYpRlg4Z1QkZWUB"
SHARDS = [f"model-0000{i}-of-00006.safetensors" for i in range(1, 7)]

def first_dir(*cands):
    for p in cands:
        if p.is_dir():
            return p
    my = Path("/content/drive/MyDrive")
    if my.is_dir():
        hits = sorted(my.glob("**/TAN"))[:8] + sorted(my.glob("**/jtagent"))[:8]
        for p in hits:
            if p.is_dir():
                return p if p.name == "TAN" else p.parent
    return None

TAN = first_dir(
    Path("/content/drive/MyDrive/TAN"),
    *sorted(Path("/content/drive/MyDrive").glob("TAN*"))[:4],
    Path("/content/drive/Shareddrives/TAN"),
    Path("/content/TAN"),
)
print("TAN", TAN)

def find_named(name):
    roots = [TAN, Path("/content/drive/MyDrive/HF_Downloads"), Path("/content/drive/MyDrive")]
    for root in roots:
        if root is None or not Path(root).exists():
            continue
        for p in [Path(root) / name, Path(root) / "jtagent" / "hf" / "full" / name]:
            if p.is_dir():
                return p
        hits = sorted(Path(root).glob(f"**/{name}"))[:8]
        if hits:
            return hits[0]
    return None

MODEL = find_named("jt-agent-model")
DATA = find_named("jt-agent-data")
print("MODEL", MODEL)
print("DATA", DATA)
if MODEL:
    for s in SHARDS:
        p = MODEL / s
        print(s, p.exists(), p.stat().st_size if p.exists() else 0)
assert MODEL is not None, "jt-agent-model folder not on mounted Drive yet"
assert DATA is not None, "jt-agent-data folder not on mounted Drive yet"
missing = [s for s in SHARDS if not (MODEL / s).is_file() or (MODEL / s).stat().st_size < 1_000_000_000]
if missing:
    print("SHARDS_PENDING", missing)
    print("GPT-2 LoRA can continue. 8B shards are still uploading to Drive.")
else:
    print("SHARDS_OK", 6)

In [ ]:
import json
from pathlib import Path

# Same Drive notebook: inventory the new 6 HF repos. Do not assert-crash if still uploading.
NEW6 = [
    ("jtagent/Qwen3.8-27B-Uncensored-MLX-bucket", "model", "Qwen3.8-27B-Uncensored-MLX-bucket"),
    ("jtagent/jt-agent-model", "model", "jtagent-jt-agent-model"),
    ("jtagent/jt-agent-data", "model", "jt-agent-data"),
    ("jtagent/jt-agent-dataset", "dataset", "jt-agent-dataset"),
    ("jtagent/chrome-browser-data", "dataset", "chrome-browser-data"),
    ("jtagent/jt-agent-data", "dataset", "jt-agent-data"),
]

def locate_folder(folder_name):
    roots = []
    if TAN is not None:
        roots.append(Path(TAN) / "jtagent" / "hf" / "full")
        roots.append(Path(TAN) / "jtagent" / "hf")
    roots.extend([
        Path("/content/drive/MyDrive/HF_Downloads"),
        Path("/content/drive/MyDrive"),
    ])
    for root in roots:
        if root is None or not Path(root).exists():
            continue
        direct = Path(root) / folder_name
        if direct.is_dir():
            return direct
        hits = sorted(Path(root).glob(f"**/{folder_name}"))[:6]
        if hits:
            return hits[0]
    return None

NEW6_REPORT = []
for repo, kind, folder in NEW6:
    path = locate_folder(folder)
    files = 0
    total = 0
    if path is not None:
        for f in path.rglob("*"):
            if f.is_file() and ".cache" not in f.parts:
                files += 1
                total += f.stat().st_size
    row = {
        "repo": repo,
        "type": kind,
        "folder": folder,
        "path": str(path) if path else None,
        "files": files,
        "bytes": total,
        "status": "ok" if files else "PENDING",
    }
    NEW6_REPORT.append(row)
    print(repo, kind, row["status"], files, total, path)

print(json.dumps(NEW6_REPORT, indent=2))
print("NEW6_PENDING", [f"{r['repo']} ({r['type']})" for r in NEW6_REPORT if r["status"] == "PENDING"])


In [ ]:
import subprocess
from pathlib import Path

REPO = Path("/content/jayti")
if not (REPO / "sandbox" / "jtagent" / "segment_jsonl.py").is_file():
    subprocess.check_call([
        "git", "clone", "--depth", "1", "-b", "main",
        "https://github.com/jaytipargal/jayti.git", str(REPO),
    ])
print("repo", REPO, (REPO / "sandbox" / "jtagent").exists())

In [ ]:
import json
import sys
from pathlib import Path

ROOT = Path("/content/TAN/jtagent")
if TAN is not None:
    ROOT = Path(TAN) / "jtagent"
ROOT.mkdir(parents=True, exist_ok=True)
sandbox = Path("/content/jayti/sandbox/jtagent")
if str(sandbox) not in sys.path:
    sys.path.insert(0, str(sandbox))

import drive_sync
import segment_jsonl
import segment_train

drive_sync.cmd_mount("1ondyw5YrwXpE6jV48nYpRlg4Z1QkZWUB", Path("/content/TAN"))
(ROOT / "hf").mkdir(parents=True, exist_ok=True)
(ROOT / "hf" / "HUB_POINTER.md").write_text(
    json.dumps({"model": str(MODEL), "dataset": str(DATA), "root": str(ROOT), "new6": NEW6_REPORT if "NEW6_REPORT" in globals() else []}, indent=2) + "\n",
    encoding="utf-8",
)
manifest = segment_jsonl.run(ROOT)
print(json.dumps(manifest, indent=2))

In [ ]:
import json
from pathlib import Path

report = segment_train.train_categories(ROOT)
push_rc = drive_sync.cmd_push("1ondyw5YrwXpE6jV48nYpRlg4Z1QkZWUB", Path(TAN) if TAN else Path("/content/TAN"))
final = {"train": report, "drive_push_rc": push_rc, "root": str(ROOT)}
(ROOT / "RUN_REPORT.md").write_text(json.dumps(final, indent=2) + "\n", encoding="utf-8")
print(json.dumps(final, indent=2))
print("JTAGENT_NOTEBOOK_DONE")